In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
    ModelRetryMiddleware,
    PIIMiddleware,
    SummarizationMiddleware,
    ToolCallLimitMiddleware,
    before_model,
)
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class DemoToolModel(FakeMessagesListChatModel):
    """A local, deterministic model that can emit predefined tool calls."""

    def bind_tools(self, tools, **kwargs):
        return self

### 1. Summarization: compress older conversation

The short trigger is only for this demo. When the message count reaches four, the fake summarizer replaces older messages with a summary and keeps the newest two messages.

In [ ]:
summary_model = FakeMessagesListChatModel(
    responses=[AIMessage(content="The demo meeting is at 3 PM.")]
)
answer_model = FakeMessagesListChatModel(
    responses=[AIMessage(content="I remember: the demo meeting is at 3 PM.")]
)

summary_agent = create_agent(
    answer_model,
    middleware=[
        SummarizationMiddleware(
            model=summary_model,
            trigger=("messages", 4),
            keep=("messages", 2),
        )
    ],
)

conversation = [
    {"role": "user", "content": "I am planning a demo meeting."},
    {"role": "assistant", "content": "What time?"},
    {"role": "user", "content": "3 PM."},
    {"role": "assistant", "content": "I will remember."},
    {"role": "user", "content": "What time is the meeting?"},
]
summary_result = summary_agent.invoke({"messages": conversation})

print("Messages retained after summarization:")
for message in summary_result["messages"]:
    print(f"{message.type}: {message.content}")
print("\nFinal answer:", summary_result["messages"][-1].content)

Messages retained after summarization:
human: Here is a summary of the conversation to date:

The demo meeting is at 3 PM.
ai: I will remember.
human: What time is the meeting?
ai: I remember: the demo meeting is at 3 PM.

Final answer: I remember: the demo meeting is at 3 PM.


### 2. Human approval: approve or reject a pretend email

`HumanInTheLoopMiddleware` pauses before `send_email` runs. The checkpointer remembers the paused state so `Command(resume=...)` can continue it. This tool only appends to a Python list; it never contacts an email service. Try changing `decision` to `"reject"` to see the email remain unsent.

In [3]:
decision = "approve"  # Change to "reject" to deny the pretend email.
email_events = []


def send_email(to: str, subject: str, body: str) -> str:
    """Record a pretend email locally; this does not send anything."""
    email_events.append({"to": to, "subject": subject, "body": body})
    return f"Demo email recorded for {to}; nothing was actually sent."


email_model = DemoToolModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "send_email",
                    "args": {
                        "to": "alex@example.com",
                        "subject": "Project update",
                        "body": "The demo is at 3 PM.",
                    },
                    "id": "demo-email-call",
                    "type": "tool_call",
                }
            ],
        ),
        AIMessage(content=f"Email decision completed: {decision}."),
    ]
)
email_agent = create_agent(
    email_model,
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                }
            }
        )
    ],
    checkpointer=InMemorySaver(),
)
email_config = {"configurable": {"thread_id": "middleware-email-demo"}}
paused = email_agent.invoke(
    {"messages": [{"role": "user", "content": "Prepare an email to Alex."}]},
    email_config,
)
print("Paused for review:", paused["__interrupt__"])
print("Email log before decision:", email_events)

email_result = email_agent.invoke(
    Command(resume={"decisions": [{"type": decision}]}),
    email_config,
)
print("Email log after decision:", email_events)
print("Agent result:", email_result["messages"][-1].content)

Paused for review: [Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'to': 'alex@example.com', 'subject': 'Project update', 'body': 'The demo is at 3 PM.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'alex@example.com', 'subject': 'Project update', 'body': 'The demo is at 3 PM.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='b95b87b93a20bf8009419904b6a43e11', response_schema=None)]
Email log before decision: []
Email log after decision: [{'to': 'alex@example.com', 'subject': 'Project update', 'body': 'The demo is at 3 PM.'}]
Agent result: Email decision completed: approve.


### 3. Call limits: control model and tool work

A model-call limit can stop an agent after a tool result, before it asks the model for another step. A tool-call limit prevents excess calls to a named tool. Here the pretend tool log makes it easy to see what ran.

In [4]:
order_lookups = []


def lookup_order(order_id: str) -> str:
    """Look up an order in this tiny in-memory demo."""
    order_lookups.append(order_id)
    return f"Order {order_id} is packed."


tool_request = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "lookup_order",
            "args": {"order_id": "A-104"},
            "id": "order-lookup-call",
            "type": "tool_call",
        }
    ],
)
limited_agent = create_agent(
    DemoToolModel(responses=[tool_request, AIMessage(content="Done.")]),
    tools=[lookup_order],
    middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
)
limited_result = limited_agent.invoke(
    {"messages": [{"role": "user", "content": "Look up order A-104."}]}
)
print("Model-call limit: tool log =", order_lookups)
print("Model-call limit: final message =", limited_result["messages"][-1].content)

order_lookups.clear()
two_tool_calls = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "lookup_order",
            "args": {"order_id": "A-104"},
            "id": "order-call-1",
            "type": "tool_call",
        },
        {
            "name": "lookup_order",
            "args": {"order_id": "B-205"},
            "id": "order-call-2",
            "type": "tool_call",
        },
    ],
)
limited_tool_agent = create_agent(
    DemoToolModel(responses=[two_tool_calls]),
    tools=[lookup_order],
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="lookup_order",
            run_limit=1,
            exit_behavior="end",
        )
    ],
)
tool_limit_result = limited_tool_agent.invoke(
    {"messages": [{"role": "user", "content": "Look up both sample orders."}]}
)
print("Tool-call limit: tool log =", order_lookups)
print("Tool-call limit: final message =", tool_limit_result["messages"][-1].content)

Model-call limit: tool log = ['A-104']
Model-call limit: final message = Model call limits exceeded: run limit (1/1)
Tool-call limit: tool log = []
Tool-call limit: final message = 'lookup_order' tool call limit reached: run limit exceeded (2/1 calls).


### 4. Retry and fallback: recover from temporary failures

`ModelRetryMiddleware` retries the same model. `ModelFallbackMiddleware` tries another model if the primary model fails. The demo models below deliberately fail, so you can see each recovery path without a network request.

In [5]:
class FailOnceModel(FakeMessagesListChatModel):
    attempts: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.attempts += 1
        if self.attempts == 1:
            raise RuntimeError("Temporary demo failure")
        return super()._generate(
            messages, stop=stop, run_manager=run_manager, **kwargs
        )


retry_model = FailOnceModel(
    responses=[AIMessage(content="Recovered on the second attempt.")]
)
retry_agent = create_agent(
    retry_model,
    middleware=[
        ModelRetryMiddleware(
            max_retries=1,
            initial_delay=0,
            jitter=False,
        )
    ],
)
retry_result = retry_agent.invoke(
    {"messages": [{"role": "user", "content": "Say hello."}]}
)
print("Retry attempts:", retry_model.attempts)
print("Retry result:", retry_result["messages"][-1].content)


class AlwaysFailModel(FakeMessagesListChatModel):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        raise RuntimeError("Primary demo model is unavailable")


primary_model = AlwaysFailModel(responses=[])
backup_model = FakeMessagesListChatModel(
    responses=[AIMessage(content="This answer came from the backup model.")]
)
fallback_agent = create_agent(
    primary_model,
    middleware=[ModelFallbackMiddleware(primary_model, backup_model)],
)
fallback_result = fallback_agent.invoke(
    {"messages": [{"role": "user", "content": "Say hello."}]}
)
print("Fallback result:", fallback_result["messages"][-1].content)

Retry attempts: 2
Retry result: Recovered on the second attempt.
Fallback result: This answer came from the backup model.


### 5. PII protection: redact before the model sees input

`PIIMiddleware` can redact detected values in incoming messages. The final state below shows the email replaced with `[REDACTED_EMAIL]`.

In [6]:
pii_agent = create_agent(
    FakeMessagesListChatModel(
        responses=[AIMessage(content="I received the redacted contact detail.")]
    ),
    middleware=[PIIMiddleware("email", strategy="redact")],
)
pii_result = pii_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Please contact sam@example.com."}
        ]
    }
)
print("Message received by the agent:", pii_result["messages"][0].content)
print("Final answer:", pii_result["messages"][-1].content)

Message received by the agent: Please contact [REDACTED_EMAIL].
Final answer: I received the redacted contact detail.


### 6. Custom hook: inspect input before a model call

Use a custom hook when built-in middleware does not fit. `@before_model` runs just before the model and here prints the latest message for learning/debugging.

In [7]:
@before_model
def show_message_before_model(state, runtime):
    latest_message = state["messages"][-1]
    print("Custom hook saw:", latest_message.type, latest_message.content)


hook_agent = create_agent(
    FakeMessagesListChatModel(
        responses=[AIMessage(content="Hello from the local demo model.")]
    ),
    middleware=[show_message_before_model],
)
hook_result = hook_agent.invoke(
    {"messages": [{"role": "user", "content": "Hi there!"}]}
)
print("Final answer:", hook_result["messages"][-1].content)

Custom hook saw: human Hi there!
Final answer: Hello from the local demo model.
